In [5]:
import pandas as pd
import numpy as np
import sys
import random

In [6]:
tissue = ['BLOOD', 'LIVER', 'LUNG', 'KIDNEY', 'all']

In [7]:
trait = pd.read_csv('/home/lucytian/data/1_Single_Cell_PRS/2_cV2F/pheno_tissue.txt', header=None)[0].tolist()

In [8]:
data_d = '/home/lucytian/data/1_Single_Cell_PRS/2_cV2F/mvp_afr_0.9_0.01/20231221/406k_geno_v2_UKB_18PCs/fit_w_val/'

In [9]:
bcount_df = pd.read_csv('/home/lucytian/group/data/gwas_geno/blood_count.tsv.gz', sep='\t', compression='gzip', low_memory=False)
bchem_df = pd.read_csv('/home/lucytian/group/data/gwas_geno/blood_biochemistry.tsv.gz', sep='\t', compression='gzip', low_memory=False)
spiro_df = pd.read_csv('/home/lucytian/group/data/gwas_geno/spirometry.tsv.gz', sep='\t', compression='gzip', low_memory=False)

In [10]:
trait_category_dict = {'INI30120': bcount_df, 'INI50030700':bchem_df, 'INI20030780':bchem_df, 'INI1003063':spiro_df, 'INI10030620': bchem_df}

In [11]:
base_dir = '/home/lucytian/data/1_Single_Cell_PRS/2_cV2F/baselines/406k_geno_v2_UKB_18PCs/fit_w_val/'

In [14]:
def generate_input(pop):
    df_n = []
    df_trait = []
    df_tissue = []
    df_sscore = []
    for tr in trait:
        base = trait_category_dict[tr][['#FID', 'IID', tr, 'population', 'split']]
        base = base[(base['population'] == pop) & (base['split'] == 'test')]
        if tr == "INI20030780":
            df_base = pd.read_csv(base_dir + '/' + tr + '/exclude_APOE/snpnet.sscore.zst', sep='\t', compression='zstd')
            df_base = df_base.rename(columns = {'exclude_APOE_SUM': tr+'_baseline'})
        else:
            df_base = pd.read_csv(base_dir + '/' + tr + '/snpnet.sscore.zst', sep='\t', compression='zstd')
            df_base = df_base.rename(columns = {tr+'_SUM': tr+'_baseline'})
        base = base.merge(df_base[['IID', tr+'_baseline']], on='IID')
        for ti in tissue:
            if tr == "INI20030780":
                df = pd.read_csv(data_d + ti + '/' + tr + '/exclude_APOE/snpnet.sscore.zst', sep='\t', compression='zstd')
                df = df.rename(columns = {'exclude_APOE_SUM': tr+'_'+ti})
            else:
                df = pd.read_csv(data_d + ti + '/' + tr + '/snpnet.sscore.zst', sep='\t', compression='zstd')
                df = df.rename(columns = {tr+'_SUM': tr+'_'+ti})
            base = base.merge(df[['IID', tr+'_'+ti]], on='IID')
        base = base.dropna()
        base = base.drop(columns=['population', 'split'])
        df_n.append(len(base))
        df_trait.append(tr)
        base.iloc[:, 2:].to_csv(tr + '_' + pop + '.tsv', sep='\t', index=False)   

In [17]:
generate_input('Afr')

,#FID,IID,INI20030780,INI20030780_baseline,INI20030780_BLOOD,INI20030780_LIVER,INI20030780_LUNG,INI20030780_KIDNEY,INI20030780_all
0,1001867,1001867,1.284262,-0.102246,-0.082622,-0.119421,-0.096366,-0.094745,-0.111781
2,1005615,1005615,1.491326,0.022218,0.020043,0.005292,0.005960,0.027418,0.004239
3,1006227,1006227,0.743840,-0.126347,-0.158122,-0.167910,-0.166979,-0.144065,-0.156024
4,1009350,1009350,1.211048,0.012438,0.019892,-0.015482,-0.002140,0.011808,-0.013524
5,1021375,1021375,0.838545,-0.018901,-0.028927,-0.032409,0.015010,-0.019112,-0.033594
...,...,...,...,...,...,...,...,...,...
1207,6009175,6009175,1.413180,0.038825,0.052031,0.043034,0.041039,0.029435,0.010044
1208,6009730,6009730,1.246745,-0.069148,-0.056622,-0.096616,-0.075643,-0.049298,-0.097321
1209,6010954,6010954,1.334738,0.007337,0.009924,-0.007368,0.037514,0.038842,0.000274
1210,6012496,6012496,1.178655,0.014089,0.008357,-0.006161,0.012352,0.029156,-0.025640
